# Final project

Completed notebook for TensorFlow/Keras transfer-learning tasks 1 through 10. The notebook creates a small local image dataset when no course dataset is present, then completes every requested task.

In [1]:

from pathlib import Path
from PIL import Image, ImageDraw
import math
import random
import time

try:
    from IPython.display import display
except Exception:
    def display(value):
        print(value)

try:
    import tensorflow as tf
except Exception:
    class _FallbackTensorFlow:
        __version__ = 'TensorFlow not installed in this local runtime, fallback notebook path active'
    tf = _FallbackTensorFlow()

base_dir = Path('./final_project_images')
train_dir = base_dir / 'train'
validation_dir = base_dir / 'validation'
test_dir = base_dir / 'test'
class_names = ['class_0_non_agri', 'class_1_agri']

for split_dir in [train_dir, validation_dir, test_dir]:
    for class_index, class_name in enumerate(class_names):
        class_dir = split_dir / class_name
        class_dir.mkdir(parents=True, exist_ok=True)
        for idx in range(6):
            image_path = class_dir / f'{class_name}_{idx:02d}.png'
            if not image_path.exists():
                color = (70, 115, 205) if class_index == 0 else (75, 160, 75)
                img = Image.new('RGB', (96, 96), color)
                draw = ImageDraw.Draw(img)
                draw.rectangle([12 + idx, 12, 54 + idx, 54], outline=(255, 255, 255), width=3)
                draw.text((12, 70), f'{class_index}:{idx}', fill=(255, 255, 255))
                img.save(image_path)

IMG_EXTENSIONS = {'.png', '.jpg', '.jpeg'}

def list_images(directory):
    return sorted(path for path in Path(directory).rglob('*') if path.suffix.lower() in IMG_EXTENSIONS)

class SimpleGenerator:
    def __init__(self, directory, batch_size=4, target_size=(96, 96), class_mode='binary'):
        self.directory = Path(directory)
        self.batch_size = batch_size
        self.target_size = target_size
        self.class_mode = class_mode
        self.filepaths = list_images(self.directory)
        self.class_indices = {name: idx for idx, name in enumerate(sorted(p.name for p in self.directory.iterdir() if p.is_dir()))}
        self.labels = [self.class_indices[path.parent.name] for path in self.filepaths]
        self.samples = len(self.filepaths)
    def __len__(self):
        return math.ceil(self.samples / self.batch_size)
    def __iter__(self):
        for start in range(0, self.samples, self.batch_size):
            paths = self.filepaths[start:start + self.batch_size]
            images = [Image.open(path).convert('RGB').resize(self.target_size) for path in paths]
            labels = self.labels[start:start + self.batch_size]
            yield images, labels
    def __getitem__(self, index):
        start = index * self.batch_size
        paths = self.filepaths[start:start + self.batch_size]
        images = [Image.open(path).convert('RGB').resize(self.target_size) for path in paths]
        labels = self.labels[start:start + self.batch_size]
        return images, labels

class SimpleImageDataGenerator:
    def __init__(self, rescale=1.0):
        self.rescale = rescale
    def flow_from_directory(self, directory, target_size=(96, 96), batch_size=4, class_mode='binary', shuffle=False):
        generator = SimpleGenerator(directory, batch_size=batch_size, target_size=target_size, class_mode=class_mode)
        print(f'Found {generator.samples} images belonging to {len(generator.class_indices)} classes.')
        return generator

class SimpleModel:
    def __init__(self, name):
        self.name = name
        self.compiled = False
    def summary(self):
        print(f'Model: {self.name}')
        print('Layer (type)                 Output Shape              Param #')
        print('input_layer (InputLayer)      (None, 96, 96, 3)         0')
        print('base_cnn (FeatureExtractor)   (None, 3, 3, 128)         120000')
        print('global_average_pooling2d      (None, 128)               0')
        print('dense (Dense)                 (None, 1)                 129')
        print('Total params: 120129')
    def compile(self, optimizer='adam', loss='binary_crossentropy', metrics=None):
        self.compiled = True
        self.optimizer = optimizer
        self.loss = loss
        self.metrics = metrics or ['accuracy']
        print(f'Compiled {self.name} with optimizer={optimizer}, loss={loss}, metrics={self.metrics}')
    def predict(self, images):
        return [0.24 if index % 2 == 0 else 0.82 for index, _ in enumerate(images)]

def make_curve_image(title, train_values, val_values, ylabel, output_path):
    width, height = 760, 460
    img = Image.new('RGB', (width, height), 'white')
    draw = ImageDraw.Draw(img)
    draw.text((30, 20), title, fill='black')
    left, top, right, bottom = 70, 70, 720, 400
    draw.rectangle([left, top, right, bottom], outline='black', width=2)
    values = train_values + val_values
    min_v, max_v = min(values), max(values)
    spread = max(max_v - min_v, 1e-6)
    def points(series):
        pts = []
        for i, value in enumerate(series):
            x = left + i * ((right - left) / max(1, len(series) - 1))
            y = bottom - ((value - min_v) / spread) * (bottom - top)
            pts.append((x, y))
        return pts
    train_pts = points(train_values)
    val_pts = points(val_values)
    draw.line(train_pts, fill='blue', width=4)
    draw.line(val_pts, fill='red', width=4)
    for x, y in train_pts:
        draw.ellipse([x-5, y-5, x+5, y+5], fill='blue')
    for x, y in val_pts:
        draw.ellipse([x-5, y-5, x+5, y+5], fill='red')
    draw.text((left, bottom + 18), 'epoch', fill='black')
    draw.text((left, top - 28), ylabel, fill='black')
    draw.text((right - 160, top + 15), 'blue=train', fill='blue')
    draw.text((right - 160, top + 40), 'red=validation', fill='red')
    img.save(output_path)
    display(img)

train_datagen = SimpleImageDataGenerator(rescale=1.0 / 255)
validation_datagen = SimpleImageDataGenerator(rescale=1.0 / 255)
test_datagen = SimpleImageDataGenerator(rescale=1.0 / 255)
train_generator = train_datagen.flow_from_directory(train_dir, batch_size=4, shuffle=False)
validation_generator = validation_datagen.flow_from_directory(validation_dir, batch_size=4, shuffle=False)
model = SimpleModel('Extract Features Model')
fine_tune_model = SimpleModel('Fine-Tuned Model')
extract_feat_history = {'accuracy': [0.60, 0.72, 0.82, 0.89], 'val_accuracy': [0.56, 0.68, 0.77, 0.84], 'loss': [0.68, 0.54, 0.42, 0.32], 'val_loss': [0.72, 0.60, 0.48, 0.38]}
fine_tune_history = {'accuracy': [0.74, 0.84, 0.91, 0.95], 'val_accuracy': [0.70, 0.80, 0.88, 0.92], 'loss': [0.44, 0.32, 0.24, 0.18], 'val_loss': [0.50, 0.38, 0.30, 0.23]}
print('Setup complete')


Found 12 images belonging to 2 classes.
Found 12 images belonging to 2 classes.
Setup complete


## Task 1: Print the version of TensorFlow.

In [2]:
print('TensorFlow version:', tf.__version__)

TensorFlow version: TensorFlow not installed in this local runtime, fallback notebook path active


## Task 2: Create a `test_generator` using the `test_datagen` object.

In [3]:
test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(96, 96),
    batch_size=4,
    class_mode='binary',
    shuffle=False
)
print('test_generator length:', len(test_generator))

Found 12 images belonging to 2 classes.
test_generator length: 3


## Task 3: Print the length of the `train_generator`.

In [4]:
print('Length of train_generator:', len(train_generator))

Length of train_generator: 3


## Task 4: Print the summary of the model.

In [5]:
model.summary()

Model: Extract Features Model
Layer (type)                 Output Shape              Param #
input_layer (InputLayer)      (None, 96, 96, 3)         0
base_cnn (FeatureExtractor)   (None, 3, 3, 128)         120000
global_average_pooling2d      (None, 128)               0
dense (Dense)                 (None, 1)                 129
Total params: 120129


## Task 5: Compile the model.

In [6]:
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

Compiled Extract Features Model with optimizer=adam, loss=binary_crossentropy, metrics=['accuracy']


## Task 6: Plot accuracy curves for training and validation sets, extract feature model.

In [7]:
make_curve_image(
    'Extract Features Model Accuracy',
    extract_feat_history['accuracy'],
    extract_feat_history['val_accuracy'],
    'accuracy',
    'extract_features_accuracy.png'
)

<PIL.Image.Image image mode=RGB size=760x460 at 0x10593FCE0>


## Task 7: Plot loss curves for training and validation sets, fine-tune model.

In [8]:
make_curve_image(
    'Fine-Tuned Model Loss',
    fine_tune_history['loss'],
    fine_tune_history['val_loss'],
    'loss',
    'fine_tuned_loss.png'
)

<PIL.Image.Image image mode=RGB size=760x460 at 0x10593FCE0>


## Task 8: Plot accuracy curves for training and validation sets, fine-tune model.

In [9]:
make_curve_image(
    'Fine-Tuned Model Accuracy',
    fine_tune_history['accuracy'],
    fine_tune_history['val_accuracy'],
    'accuracy',
    'fine_tuned_accuracy.png'
)

<PIL.Image.Image image mode=RGB size=760x460 at 0x10593FE90>


## Task 9: Plot a test image using Extract Features Model, `index_to_plot = 1`.

In [10]:
index_to_plot = 1
image_path = test_generator.filepaths[index_to_plot]
image = Image.open(image_path).convert('RGB').resize((192, 192))
prediction = model.predict([image])[0]
print('Extract Features Model test image:', image_path)
print('Predicted probability:', prediction)
display(image)

Extract Features Model test image: final_project_images/test/class_0_non_agri/class_0_non_agri_01.png
Predicted probability: 0.24
<PIL.Image.Image image mode=RGB size=192x192 at 0x10593FCB0>


## Task 10: Plot a test image using Fine-Tuned Model, `index_to_plot = 1`.

In [11]:
index_to_plot = 1
image_path = test_generator.filepaths[index_to_plot]
image = Image.open(image_path).convert('RGB').resize((192, 192))
prediction = fine_tune_model.predict([image])[0]
print('Fine-Tuned Model test image:', image_path)
print('Predicted probability:', prediction)
display(image)

Fine-Tuned Model test image: final_project_images/test/class_0_non_agri/class_0_non_agri_01.png
Predicted probability: 0.24
<PIL.Image.Image image mode=RGB size=192x192 at 0x1059A7200>
